# FOCUS-3D Inference Demo

This notebook demonstrates how to run FOCUS-3D inference.

It includes two common use cases:

1. Run inference on a single 3D microscopy volume.
2. Run batch inference for all volumes in a folder.

The input image should be a 3D volume with shape `(Z, Y, X)`, saved as `.tif`, `.tiff`, or `.zarr`.

The output includes:

- an instance segmentation map;
- a confidence map;
- a JSON log file recording inference parameters and runtime information.

In [ ]:
# ============================================================
# Cell 1. Environment and imports
# ============================================================

import importlib
import json
import os
import platform
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile
from skimage.segmentation import find_boundaries

# Set GPU before importing the inference backend.
# If you change this value, restart the notebook kernel.
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

FOCUS3D_BACKEND_DIR = (
    Path.cwd() / '../src/focus3d/segmentation/FOCUS3D'
).resolve()

if str(FOCUS3D_BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(FOCUS3D_BACKEND_DIR))

if platform.system().lower() == 'windows':
    inference_module = importlib.import_module('inference_win')
else:
    inference_module = importlib.import_module('inference')

infer_volume = inference_module.infer_volume
infer_folder = inference_module.infer_folder

print('FOCUS-3D inference notebook is ready.')
print('CUDA_VISIBLE_DEVICES =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('Backend path:', FOCUS3D_BACKEND_DIR)
print('Inference module:', inference_module.__name__)

## Common configuration

Set the model config, checkpoint, and inference parameters here.

For most 3D cell segmentation datasets, the default patch size `(32, 96, 96)` and stride `(24, 64, 64)` provide a good balance between accuracy and memory usage.

Important parameters:

- `z_ratio`: relative voxel spacing along Z compared with XY.
- `background_threshold`: raw intensity threshold used to skip near-empty patches.
- `batch_size`: number of patches processed per GPU batch.
- `score_thresh`: confidence threshold for predicted instances.
- `mask_thresh`: mask probability threshold.
- `min_edge_area`: minimum 2D connected component area used in post-processing.

In [ ]:
# ============================================================
# Cell 2. Common configuration
# ============================================================

CONFIG_FILE = FOCUS3D_BACKEND_DIR / 'configs' / '3d_test.yaml'
WEIGHTS_PATH = FOCUS3D_BACKEND_DIR / 'model_final.pth'

CONFIG_FILE = str(CONFIG_FILE)
WEIGHTS_PATH = str(WEIGHTS_PATH)

print('Config file:', CONFIG_FILE)
print('Weights:', WEIGHTS_PATH)

# Common inference parameters.
# These parameters are shared by single-volume inference and batch inference.
INFER_PARAMS = {
    'z_ratio': 5.0,
    'lower_percentile': 1.0,
    'upper_percentile': 99.0,
    'background_threshold': 0.0,
    'patch_size': [32, 96, 96],
    'stride': [24, 64, 64],
    'batch_size': 12,
    'score_thresh': 0.7,
    'mask_thresh': 0.5,
    'min_edge_area': 64,
    'topk_postprocess': 300,
    'save_intermediate': False,
}

print('Config file:', CONFIG_FILE)
print('Weights:', WEIGHTS_PATH)
print('Inference parameters:')
print(json.dumps(INFER_PARAMS, indent=2))

## Single-volume inference

In this section, we run FOCUS-3D on one 3D microscopy volume.

The result will be saved to `OUTPUT_DIR`.

In [ ]:
# ============================================================
# Cell 3. Single-volume inference
# ============================================================

PROJECT_ROOT = Path.cwd().parent

IMAGE_PATH = PROJECT_ROOT / 'example' / 'data' / '00000.tif'
OUTPUT_DIR = PROJECT_ROOT / 'example' / 'output'

if not IMAGE_PATH.exists():
    raise FileNotFoundError(f'Input image not found: {IMAGE_PATH}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Input image:', IMAGE_PATH)
print('Output dir:', OUTPUT_DIR)

infer_result = infer_volume(
    image_path=IMAGE_PATH,
    config_file=CONFIG_FILE,
    weights_path=WEIGHTS_PATH,
    output_dir=OUTPUT_DIR,
    **INFER_PARAMS,
)

print('Single-volume inference finished.')
print('Instance map:', infer_result['instance_map_path'])
print('Confidence map:', infer_result['confidence_map_path'])
print('Log file:', infer_result['log_json_path'])
print('Number of inferred patches:', infer_result['num_infer_patches'])
print(
    'Number of skipped background patches:',
    infer_result['num_skipped_background_patches'],
)

## Visualize the segmentation result

The following cells display representative Z slices from the 3D output.

We show:

1. the raw image;
2. the instance segmentation map;
3. an overlay of segmentation boundaries on the raw image.

By default, the notebook visualizes the middle Z slice. It also provides an option to show the slice with the largest foreground area, which is often useful for quickly checking segmentation quality.

In [ ]:
# ============================================================
# Cell 4. Visualization helper functions
# ============================================================


def normalize_for_display(image_2d, p_low=1, p_high=99):
    """Normalize a 2D image for display using robust percentiles."""
    image_2d = np.asarray(image_2d, dtype=np.float32)

    lo, hi = np.percentile(image_2d, [p_low, p_high])
    if hi <= lo:
        return np.zeros_like(image_2d, dtype=np.float32)

    image_2d = np.clip(image_2d, lo, hi)
    image_2d = (image_2d - lo) / (hi - lo)
    return image_2d


def choose_middle_slice(volume):
    """Return the middle Z index of a 3D volume."""
    return int(volume.shape[0] // 2)


def choose_foreground_rich_slice(instance_map):
    """
    Return the Z index with the largest number of foreground voxels.
    This is useful when the middle slice contains little or no tissue.
    """
    fg_per_z = np.count_nonzero(instance_map > 0, axis=(1, 2))
    return int(np.argmax(fg_per_z))


def show_segmentation_slice(
    raw_volume,
    instance_map,
    z=None,
    title_prefix='',
    show_boundaries=True,
):
    """
    Show raw image, instance map, and boundary overlay for one Z slice.
    """
    if raw_volume.shape != instance_map.shape:
        raise ValueError(
            f'raw_volume and instance_map must have the same shape. '
            f'Got {raw_volume.shape} and {instance_map.shape}.'
        )

    if z is None:
        z = choose_middle_slice(raw_volume)

    raw_slice = raw_volume[z]
    seg_slice = instance_map[z]

    raw_disp = normalize_for_display(raw_slice)
    boundaries = find_boundaries(seg_slice, mode='outer')

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    axes[0].imshow(raw_disp, cmap='gray')
    axes[0].set_title(f'{title_prefix}Raw image, Z={z}')
    axes[0].axis('off')

    axes[1].imshow(seg_slice, cmap='nipy_spectral', interpolation='nearest')
    axes[1].set_title(f'{title_prefix}Instance map, Z={z}')
    axes[1].axis('off')

    axes[2].imshow(raw_disp, cmap='gray')
    if show_boundaries:
        axes[2].imshow(
            np.ma.masked_where(~boundaries, boundaries),
            cmap='autumn',
            alpha=0.9,
        )
    axes[2].set_title(f'{title_prefix}Boundary overlay, Z={z}')
    axes[2].axis('off')

    plt.tight_layout()
    plt.show()

    unique_ids = np.unique(seg_slice)
    unique_ids = unique_ids[unique_ids > 0]

    print(f'Z slice: {z}')
    print(f'Number of instances visible in this slice: {len(unique_ids)}')
    print(
        f'Foreground pixels in this slice: {int(np.count_nonzero(seg_slice > 0))}'
    )

In [ ]:
# ============================================================
# Cell 5. Show the middle Z slice
# ============================================================

raw_volume = tifffile.imread(str(IMAGE_PATH))
instance_map = tifffile.imread(infer_result['instance_map_path'])
confidence_map = tifffile.imread(infer_result['confidence_map_path'])

raw_volume = np.squeeze(raw_volume)
instance_map = np.squeeze(instance_map)

z_middle = choose_middle_slice(raw_volume)

show_segmentation_slice(
    raw_volume=raw_volume,
    instance_map=instance_map,
    z=z_middle,
    title_prefix='Middle slice: ',
)

## Batch inference for a folder

This section runs inference on all `.tif`, `.tiff`, and `.zarr` files in a folder.

The batch function saves only the final instance map for each input volume

In [ ]:
# ============================================================
# Cell 6. Run batch inference
# ============================================================

# Project root:
#   focus3d/
#
# Input folder:
#   focus3d/example/data
#
# Batch output folder:
#   focus3d/example/output_batch

PROJECT_ROOT = Path.cwd().parent

INPUT_DIR = PROJECT_ROOT / 'example' / 'data'
BATCH_OUTPUT_DIR = PROJECT_ROOT / 'example' / 'output_batch'

if not INPUT_DIR.exists():
    raise FileNotFoundError(f'Input folder does not exist: {INPUT_DIR}')

BATCH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Input folder:', INPUT_DIR)
print('Batch output dir:', BATCH_OUTPUT_DIR)

BATCH_INFER_PARAMS = INFER_PARAMS.copy()
BATCH_INFER_PARAMS.pop('save_intermediate', None)

batch_summary = infer_folder(
    input_dir=INPUT_DIR,
    output_dir=BATCH_OUTPUT_DIR,
    config_file=CONFIG_FILE,
    weights_path=WEIGHTS_PATH,
    recursive=False,
    skip_existing=True,
    keep_failed_tmp=False,
    **BATCH_INFER_PARAMS,
)

print('\nBatch inference finished.')
print('Total:', batch_summary['total'])
print('Done:', len(batch_summary['done']))
print('Skipped:', len(batch_summary['skipped']))
print('Failed:', len(batch_summary['failed']))